# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/innouguru/flyrank-intenship-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata

# Retrieve the Hugging Face token stored in Colab Secrets
HF_TOKEN = userdata.get("HF_TOKEN")

# Check whether the token was successfully loaded
print("Token loaded:", HF_TOKEN is not None)

Token loaded: True


In [2]:
from datasets import load_dataset_builder

# Load the dataset builder for the specified FlyRank warehouse table
builder = load_dataset_builder(
    "FlyRank/internship-warehouse",         # Hugging Face dataset repository
    "fact_content_daily_performance",       # Table to inspect
    token=HF_TOKEN                          # Hugging Face access token for authentication
)

# Display the dataset's features (column names and their data types)
print(builder.info.features)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

{'report_date': Value('date32'), 'client_hash_id': Value('string'), 'content_hash_id': Value('string'), 'client_has_gsc': Value('bool'), 'client_has_ga4': Value('bool'), 'gsc_data_available': Value('bool'), 'ga4_data_available': Value('bool'), 'gsc_impressions': Value('int64'), 'gsc_clicks': Value('int64'), 'gsc_sum_position': Value('int64'), 'gsc_avg_position': Value('float64'), 'ga4_pageviews': Value('int64'), 'ga4_sessions': Value('int64'), 'ga4_users': Value('int64'), 'ga4_engaged_sessions': Value('int64'), 'ga4_total_engagement_sec': Value('int64'), 'sessions_organic': Value('int64'), 'sessions_direct': Value('int64'), 'sessions_referral': Value('int64'), 'sessions_social': Value('int64'), 'sessions_paid': Value('int64'), 'sessions_ai': Value('int64'), 'ai_chatgpt': Value('int64'), 'ai_perplexity': Value('int64'), 'ai_gemini': Value('int64'), 'ai_copilot': Value('int64'), 'ai_claude': Value('int64'), 'ai_meta': Value('int64'), 'ai_other': Value('int64'), 'scroll_events': Value('in

In [3]:
import duckdb

# Create an in-memory DuckDB connection.
con = duckdb.connect()

# Give DuckDB permission to access the gated Hugging Face warehouse.
con.execute(
    f"""CREATE SECRET (
        TYPE huggingface,
        TOKEN '{HF_TOKEN}'
    )"""
)

# Location of the FlyRank internship warehouse.
rel = "hf://datasets/FlyRank/internship-warehouse"

In [4]:
# Confirm that the warehouse table can be accessed.

con.sql(
    f"""
    SELECT COUNT(*)
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet'
    )
    """
)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     78835655 │
└──────────────┘

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [5]:
# The five observed performance signals used as features.
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

# Build the feature frame directly from the March daily warehouse data.
# Each row represents one client-content observation on one day.
#
# We filter to rows where both GSC and GA4 data are available.
# No imputation is applied because our earlier check showed
# zero missing values for these five fields after this filter.

feature_df = con.sql(
    f"""
    SELECT
        client_hash_id,
        content_hash_id,
        report_date,

        gsc_impressions,
        gsc_clicks,
        gsc_avg_position,
        ga4_pageviews,
        ga4_engaged_sessions

    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )

    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    """
).df()


# Select only the five features for the model.
X = feature_df[features].copy()

print("Feature vector shape:", X.shape)
print("\nMissing values:")
print(X.isna().sum())

X.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature vector shape: (364347, 5)

Missing values:
gsc_impressions         0
gsc_clicks              0
gsc_avg_position        0
ga4_pageviews           0
ga4_engaged_sessions    0
dtype: int64


,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions
0,5,0,5.400000,1,0
1,39,0,5.666667,2,0
2,179,0,5.156425,2,0
3,72,0,7.694444,1,0
4,3282,1,6.167885,1,0


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

### Feature notes

* **`gsc_impressions`** — Measures how often the content appeared in Google search results. No missing value.No missing-value imputation is applied after filtering to rows where both GSC and GA4 data are available. It is numeric, not categorical, and represents an observed historical signal available before the review/refresh decision.

* **`gsc_clicks`** — Measures clicks from Google search results to the content. No missing-value imputation is applied after filtering to rows where both GSC and GA4 data are available. It is numeric, not categorical, and represents an observed historical signal available before the review/refresh decision.

* **`gsc_avg_position`** — Measures the content's average position in Google search results. No missing-value imputation is applied after filtering to rows where both GSC and GA4 data are available. It is numeric, not categorical, and represents an observed historical signal available before the review/refresh decision.

* **`ga4_pageviews`** — Measures recorded pageviews for the content. No missing-value imputation is applied after filtering to rows where both GSC and GA4 data are available. It is numeric, not categorical, and represents an observed historical signal available before the review/refresh decision.

* **`ga4_engaged_sessions`** — Measures sessions that engaged with the content. No missing-value imputation is applied after filtering to rows where both GSC and GA4 data are available. It is numeric, not categorical, and represents an observed historical signal available before the review/refresh decision.


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [6]:
# Build the client-content feature and label frame used for the
# leakage experiment.
#
# The daily warehouse data is first converted into percentile ranks
# and combined into a daily performance score.
#
# The trend of that score is then used to create the proxy label.
# The five raw performance signals are averaged to match the
# client-content grain of the label.

analysis_df = con.sql(
    f"""
    WITH daily_data AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position,
            ga4_pageviews,
            ga4_engaged_sessions
        FROM read_parquet(
            '{rel}/fact_content_daily_performance/**/*.parquet',
            hive_partitioning = true
        )
        WHERE month = '2026-03'
          AND gsc_data_available IS TRUE
          AND ga4_data_available IS TRUE
    ),

    ranked_daily AS (
        SELECT
            *,
            PERCENT_RANK() OVER (
                PARTITION BY report_date
                ORDER BY gsc_impressions
            ) AS impressions_rank,

            PERCENT_RANK() OVER (
                PARTITION BY report_date
                ORDER BY gsc_clicks
            ) AS clicks_rank,

            1 - PERCENT_RANK() OVER (
                PARTITION BY report_date
                ORDER BY gsc_avg_position
            ) AS position_rank,

            PERCENT_RANK() OVER (
                PARTITION BY report_date
                ORDER BY ga4_pageviews
            ) AS pageviews_rank,

            PERCENT_RANK() OVER (
                PARTITION BY report_date
                ORDER BY ga4_engaged_sessions
            ) AS engagement_rank

        FROM daily_data
    ),

    scored_daily AS (
        SELECT
            *,
            (
                impressions_rank
                + clicks_rank
                + position_rank
                + pageviews_rank
                + engagement_rank
            ) / 5 AS performance_score
        FROM ranked_daily
    ),

    trends AS (
        SELECT
            client_hash_id,
            content_hash_id,
            REGR_SLOPE(
                performance_score,
                DATE_DIFF(
                    'day',
                    DATE '2026-03-01',
                    report_date
                )
            ) AS performance_trend
        FROM scored_daily
        GROUP BY
            client_hash_id,
            content_hash_id
        HAVING COUNT(*) >= 2
    ),

    features AS (
        SELECT
            client_hash_id,
            content_hash_id,
            AVG(gsc_impressions) AS gsc_impressions,
            AVG(gsc_clicks) AS gsc_clicks,
            AVG(gsc_avg_position) AS gsc_avg_position,
            AVG(ga4_pageviews) AS ga4_pageviews,
            AVG(ga4_engaged_sessions) AS ga4_engaged_sessions
        FROM daily_data
        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        f.*,
        t.performance_trend,
        t.performance_trend < 0 AS is_declining_label

    FROM features AS f

    INNER JOIN trends AS t
        ON f.client_hash_id = t.client_hash_id
        AND f.content_hash_id = t.content_hash_id
    """
).df()

print("Analysis shape:", analysis_df.shape)
analysis_df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Analysis shape: (45066, 9)


,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_pageviews,ga4_engaged_sessions,performance_trend,is_declining_label
0,client_65de48885f4ef01b,content_b1f61fc81b28b2d4,50.888889,0.222222,4.418032,1.777778,0.000000,-0.000562,True
1,client_65de48885f4ef01b,content_e25ea7297a1dffd3,157.720000,0.920000,4.392897,2.360000,0.040000,-0.002863,True
2,client_65de48885f4ef01b,content_3c286ded8bd68120,114.736842,0.789474,8.439390,1.631579,0.105263,0.000559,False
3,client_65de48885f4ef01b,content_b2108e8fe3360fa6,35.928571,0.571429,5.531459,1.642857,0.071429,0.006169,False
4,client_65de48885f4ef01b,content_ff867882e604fa96,12.000000,0.000000,2.850000,1.000000,0.000000,-0.000397,True


HONEST MODEL

In [7]:
from sklearn.tree import DecisionTreeClassifier
import numpy as np

# These are the only legitimate features.
features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

# Build X from observable performance signals only.
X_honest = (
    analysis_df[features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

# The proxy outcome we are trying to identify.
y = analysis_df["is_declining_label"]


# Train a deliberately simple tree for the quick leakage check.
honest_tree = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

honest_tree.fit(X_honest, y)


# Precision@50: among the 50 highest-scored pages,
# how many are actually labelled as declining?
def precision_at_k(scores, y_true, k=50):
    top_k = np.argsort(scores)[-k:]
    return np.mean(np.asarray(y_true)[top_k])


honest_scores = honest_tree.predict_proba(X_honest)[:, 1]

honest_precision = precision_at_k(
    honest_scores,
    y,
    50
)

print(f"Honest tree Precision@50: {honest_precision:.3f}")

Honest tree Precision@50: 0.700


LEAKY MODEL

In [8]:
# Deliberately introduce the label-derived performance_trend
# to demonstrate leakage.

leaky_features = features + ["performance_trend"]

X_leaky = (
    analysis_df[leaky_features]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

leaky_tree = DecisionTreeClassifier(
    max_depth=2,
    class_weight="balanced",
    random_state=42
)

leaky_tree.fit(X_leaky, y)

leaky_scores = leaky_tree.predict_proba(X_leaky)[:, 1]

leaky_precision = precision_at_k(
    leaky_scores,
    y,
    50
)

print(f"Leaky tree Precision@50: {leaky_precision:.3f}")

Leaky tree Precision@50: 1.000


Feature Window Test

In [9]:
# Future-window leakage test.
# Check whether any observation used to build our five features
# occurs after the stated decision moment: March 31, 2026.

future_feature_rows = con.sql(
    f"""
    SELECT
        COUNT(*) AS future_rows
    FROM read_parquet(
        '{rel}/fact_content_daily_performance/**/*.parquet',
        hive_partitioning = true
    )
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
      AND report_date > DATE '2026-03-31'
    """
).fetchone()[0]

print("Feature observations after decision moment:", future_feature_rows)

Feature observations after decision moment: 0


Product flags

In [10]:
# Check the actual feature set for product-generated decision flags.
# Our intended features should be measured GSC/GA4 signals,
# not precomputed recommendations or health/decision flags.

selected_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_engaged_sessions"
]

print("Selected features:")
for feature in selected_features:
    print("-", feature)

print("\nExcluded decision/label-derived fields:")
print("- performance_trend")
print("- is_declining_label")

Selected features:
- gsc_impressions
- gsc_clicks
- gsc_avg_position
- ga4_pageviews
- ga4_engaged_sessions

Excluded decision/label-derived fields:
- performance_trend
- is_declining_label


I attacked the feature set in three ways.

Label-derived information: I first trained a quick tree using the five observed GSC/GA4 features and obtained a Precision@50 of 0.700. I then deliberately added performance_trend, which is used to derive is_declining_label. Precision@50 increased to 1.000. This jump demonstrates leakage because the added feature contains the information used to construct the outcome. I therefore excluded it from the honest feature set.

Future-window information: The decision moment is defined as after March 31, 2026. I checked the feature observations for dates after that point and found 0 observations. Therefore, no post-decision observations entered the feature window used in this experiment.

Product flags: I audited the selected feature set. The five features are measured GSC/GA4 performance signals. No product-generated decision flag was included. The label-derived fields performance_trend and is_declining_label were kept outside the honest feature set.

The final honest feature set therefore contains only the five observed performance signals.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

* **`performance_trend`** — Excluded because it is used to derive the proxy label and therefore leaks outcome information.

* **`is_declining_label`** — Excluded because it is the outcome being identified, not an input feature.

* **`report_date`** — Excluded as a predictive feature because it defines the observation window rather than describing content performance.

* **`client_hash_id`** — Excluded because it is an identifier, not a content-performance signal.

* **`content_hash_id`** — Excluded because it is an identifier, not a content-performance signal.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.